# Phase 3: Advanced Optimization & Kaggle Submission

Welcome to the final phase of our Customer Churn Prediction project. In this notebook, we will push our model's performance beyond the baseline by using advanced techniques like Optuna for hyperparameter tuning and Stratified K-Fold Cross-Validation.

## Objective
Our goal is to achieve the highest possible ROC-AUC score and prepare a robust final submission for the Kaggle competition.

## Notebook Structure
1. **Setup & Data Loading**: Preparing our environment.
2. **Advanced Feature Engineering**: Refining our input variables.
3. **Hyperparameter Tuning with Optuna**: Automated optimization of XGBoost.
4. **K-Fold Cross-Validation**: Robust performance evaluation.
5. **Final Training & Submission**: Generating the Kaggle file.

## 1. Setup & Data Loading

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Set visualization theme
sns.set_theme(style="whitegrid", palette="muted")
print("Environment Setup Complete.")

Environment Setup Complete.


In [12]:
# Load data
train_path = '../Dataset/train.csv'
test_path = '../Dataset/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Data Loaded. Train: {train_df.shape[0]} rows, Test: {test_df.shape[0]} rows.")

Data Loaded. Train: 594194 rows, Test: 254655 rows.


## 2. Advanced Feature Engineering

Based on our EDA and baseline, we'll implement some refined features.

In [13]:
def apply_fe(df):
    # Convert TotalCharges to numeric
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    # Fill missing TotalCharges with MonthlyCharges * tenure
    df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])
    
    # Service Count
    services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['ServiceCount'] = (df[services] != 'No').sum(axis=1)
    
    # Calculated Average Monthly Spend
    df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
    
    # Calculated Tenure (often useful in synthetic datasets)
    df['CalculatedTenure'] = df['TotalCharges'] / (df['MonthlyCharges'] + 0.01)
    
    # Internet presence
    df['HasInternet'] = (df['InternetService'] != 'No').astype(int)
    
    # Drop irrelevant columns
    cols_to_drop = ['id', 'gender']
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

train_df = apply_fe(train_df)
test_df = apply_fe(test_df)

print("Feature Engineering Applied.")

Feature Engineering Applied.


In [14]:
# Encoding
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in cat_cols: cat_cols.remove('Churn')

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

# Encode target
le_target = LabelEncoder()
train_df['Churn'] = le_target.fit_transform(train_df['Churn'])

print("Categorical Encoding Complete.")

Categorical Encoding Complete.


## 3. Hyperparameter Tuning with Optuna

We'll use Optuna to search for a better set of hyperparameters. We've defined an objective function that evaluates the ROC-AUC using a simple train-validation split.

In [15]:
X = train_df.drop(columns=['Churn'])
y = train_df['Churn']

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'early_stopping_rounds': 50,
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1
    }
    
    X_t, X_v, y_t, y_v = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    model = XGBClassifier(**params)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
    
    preds = model.predict_proba(X_v)[:, 1]
    return roc_auc_score(y_v, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Study Complete.")
print(f"Best Score: {study.best_value}")
print(f"Best Params: {study.best_params}")

[I 2026-03-13 23:00:53,611] A new study created in memory with name: no-name-58d5ab66-48b9-4ea8-b2ff-710bfe765868
[I 2026-03-13 23:02:57,519] Trial 0 finished with value: 0.9160242361680083 and parameters: {'n_estimators': 781, 'max_depth': 10, 'learning_rate': 0.011863654543366103, 'subsample': 0.9079253141624225, 'colsample_bytree': 0.7799596028362481, 'gamma': 3.2944939088439003, 'min_child_weight': 3}. Best is trial 0 with value: 0.9160242361680083.
[I 2026-03-13 23:03:51,862] Trial 1 finished with value: 0.9165483378169165 and parameters: {'n_estimators': 674, 'max_depth': 10, 'learning_rate': 0.05191175736297665, 'subsample': 0.5836980193350672, 'colsample_bytree': 0.511792276352262, 'gamma': 2.424634417748897, 'min_child_weight': 8}. Best is trial 1 with value: 0.9165483378169165.
[I 2026-03-13 23:05:20,734] Trial 2 finished with value: 0.9166127593606737 and parameters: {'n_estimators': 1262, 'max_depth': 8, 'learning_rate': 0.044316930937732826, 'subsample': 0.7843533038449233

Study Complete.
Best Score: 0.9169829829211543
Best Params: {'n_estimators': 1132, 'max_depth': 6, 'learning_rate': 0.04066165970612231, 'subsample': 0.5841340706897473, 'colsample_bytree': 0.5710355301448193, 'gamma': 1.9102933732119212, 'min_child_weight': 7}


## 4. K-Fold Cross-Validation

Now we'll use the best parameters to train across 5 folds for even better stability and performance.

In [16]:
best_params = study.best_params
best_params['eval_metric'] = 'auc'
best_params['early_stopping_rounds'] = 50

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_df))
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    # OOF Predictions
    fold_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = fold_preds
    
    # Test Predictions (Averaging)
    test_preds += model.predict_proba(test_df)[:, 1] / skf.n_splits
    
    score = roc_auc_score(y_val, fold_preds)
    scores.append(score)
    print(f"Fold {fold + 1} ROC-AUC: {score:.5f}")

print("-" * 30)
print(f"Mean ROC-AUC Score: {np.mean(scores):.5f}")

Fold 1 ROC-AUC: 0.91618
Fold 2 ROC-AUC: 0.91730
Fold 3 ROC-AUC: 0.91675
Fold 4 ROC-AUC: 0.91791
Fold 5 ROC-AUC: 0.91497
------------------------------
Mean ROC-AUC Score: 0.91662


## 5. Final Submission

Finally, we'll format our predictions and save the CSV for Kaggle.

In [19]:
test_preds

array([0.07636511, 0.00041946, 0.09324003, ..., 0.24283182, 0.00204443,
       0.40392241], shape=(254655,))

In [34]:
print(test_df.columns)

Index(['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'ServiceCount', 'AvgMonthlySpend', 'CalculatedTenure',
       'HasInternet'],
      dtype='object')


In [38]:
print(test_df.columns.tolist())

['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'AvgMonthlySpend', 'CalculatedTenure', 'HasInternet']


In [40]:
import pandas as pd

# Load the raw test dataset (with id column intact)
raw_test = pd.read_csv(r"E:\CraftSpace\Playground_Series\S6E3\Dataset\test.csv")

# Preserve IDs
test_ids = raw_test["id"].copy()

# Apply feature engineering to test set (this will drop id)
test_df = apply_fe(raw_test)

# Build submission with original IDs + predictions
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

# Save under Own_Notebook folder
output_path = r"E:\CraftSpace\Playground_Series\S6E3\Own_Notebook\submission.csv"
submission.to_csv(output_path, index=False)

print(f"Submission file created successfully at: {output_path}")

Submission file created successfully at: E:\CraftSpace\Playground_Series\S6E3\Own_Notebook\submission.csv
